Delegative Reinforcement Learning.

THis is a sequential decision making process which involves evolving belief over multiple world models. It triggers when action is unsafe under any belief model and delegates to human. This ensure acting safely in multi step and high stake settings. 

At every step the agent think " if it should act, or if it should ask at that step to avoid potential decisions later."

THis cocnept looks similar to Learning to Defer on a high level idea because both model "defer/delegate to human when uncertain instead of making a potentially harmful decision."

But under the hood, they're different in the way they define uncertainity, or how they use human, and the kinds of problems they're solving. 


l2D is classifiaction where as Delegative reinforcement learning is a seqence of decisions. 

And as a RL, its main goal is to maximize long term award while avoiding traps, where as the objective of L2D would be to minimize classification loss. 

Human is treated a reliable fall back policy in DRL and is usually a better classifier than AI, like an oracle in L2D


In [ ]:
import gym
from gym import spaces
import numpy as np
import random

State = int
Action = int

class SimpleTrapEnv(gym.Env):
    def __init__(self):
        # States: 0=start,1=middle,2=goal,3=trap
        self.observation_space = spaces.Discrete(4)
        self.action_space = spaces.Discrete(2)  # 0=left,1=right
        self.state = 0
    def reset(self):
        self.state = 0
        return self.state
    def step(self, action):
        if self.state == 0:
            self.state = 1 if action == 1 else 3  # safe vs trap
        elif self.state == 1:
            self.state = 2
        done = self.state in [2,3]
        reward = 1.0 if self.state == 2 else -10.0 if self.state == 3 else 0.0
        return self.state, reward, done, {}


In [ ]:
# Two models with different transition probabilities in state 0:
models = [
    {0: {0: 0.9, 1:0.1}},  # model#0 thinks action 0 is usually safe
    {0: {0: 0.1, 1:0.9}},  # model#1 thinks action 1 is safer
]
belief = np.array([0.5, 0.5])

def sample_model():
    return np.random.choice([0,1], p=belief)


In [ ]:
def unsafe(model_idx, state, action, threshold=0.3):
    probs = models[model_idx][state]
    p = probs.get(action, 0)
    return p < threshold


In [ ]:
def advisor_policy(state):
    return 1  # always go right—safe path


In [ ]:
env = SimpleTrapEnv()
N_EPISODES = 100
for ep in range(N_EPISODES):
    state = env.reset()
    done = False
    model_idx = sample_model()
    while not done:
        action = advisor_policy(state) if unsafe(model_idx, state, action:=env.action_space.sample()) else action
        nxt, rew, done, _ = env.step(action)
        # Very basic belief update: reward >0 increases credibility of chosen model
        if rew > 0:
            belief[model_idx] += 0.1
            belief /= belief.sum()
        state = nxt
